In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [15]:
def generate_env(beta2, beta3, sig, mult=1):
    x1 = np.random.normal(0, sig) * mult
    x2 = x1 + np.random.normal(0, sig) * mult
    x3 = 0.3 * x1 + np.random.normal(0, sig) * mult
    x4 = 0.2 * x3 + np.random.normal(0, sig) * mult
    y = beta2 * x3 + beta3 * x4 + np.random.normal(0, sig) * mult
    x5 = 0.1 * x2 + 0.3 * y + np.random.normal(0, sig) * mult
    x6 = 0.5 * y + np.random.normal(0, sig) * mult

    return [x1, x2, x3, x4, x5, x6, y]

def generate_envs(num, samples, sig, random_noise=False):
    betas = np.random.uniform(-10, 10, size=(num, 2))

    envs = np.zeros(shape=(num, samples, 7))
    for i in range(num):
        for j in range(samples):
            noise_multiplier = 1
            if random_noise: noise_multiplier = np.random.uniform(1, 10)

            xs = generate_env(betas[i][0], betas[i][1], sig, noise_multiplier)
            envs[i][j] = xs 

    return envs 

In [ ]:
def compute_statistic(data, S):
    residuals = []
    for env in data:
        model = LinearRegression()
        model.fit(env[:, :-1][:, S], env[:, -1])
        r = np.sum((model.predict(env[:, :-1][:, S]) - env[:, -1]) ** 2)

        residuals.append(r)

    Ts = np.min(residuals) / np.max(residuals)
    return Ts


def draw_from_distribution(rank, x_sizes):
    zs = []
    for ne in x_sizes:
        zs.append(np.random.chisquare(ne - rank))

    return np.min(zs) / np.max(zs)

def monte_carlo_test(T, rank, x_sizes, steps=1000):
    bigger = 0
    for i in range(steps):
        if T > draw_from_distribution(rank, x_sizes):
            bigger += 1

    return bigger / steps

    
def test(T, S, alpha, x_sizes):
    rank = np.sum(S)
    probability = monte_carlo_test(T, rank, x_sizes)
    # print(probability)
    if probability > alpha:
        return True
    return False


def all_boolean_vectors(n):
    return ((np.arange(2**n)[:, None] >> np.arange(n - 1, -1, -1)) & 1).astype(bool)


def LICP(data, alpha=0.1):
    x_sizes = [d.shape[0] for d in data]

    all_S = all_boolean_vectors(6)
    S_approx = []

    for S in all_S:
        if np.sum(S) == 0:
            continue
        T = compute_statistic(data, S)
        if test(T, S, alpha, x_sizes):
            if len(S_approx) == 0:
                S_approx = S 
            else:
                S_approx = S_approx & S

    return S_approx


In [18]:
ground_truth = [False, False, True, True, False, False]

total = 110
bad = 0
all_false = 0

skipped = 0
for i in range(total):
    data = generate_envs(5, 100, 0.01)
    res = LICP(data, alpha=0.1)

    if len(res) == 0:
        skipped += 1
        continue

    if np.sum(res) == 0:
        all_false += 1

    is_subset = np.all(~res | ground_truth)

    # print(f"{i}: {res} - is subset: {is_subset}")    

    if not is_subset:
        bad += 1
    

print(f"False Positive: {bad / (total - skipped)}")
print(f"False Negative: {all_false / (total - skipped)}")

False Positive: 0.03636363636363636
False Negative: 0.8181818181818182


In [17]:
ground_truth = [False, False, True, True, False, False]

total = 110
bad = 0
all_false = 0

skipped = 0
for i in range(total):
    data = generate_envs(5, 100, 0.01, True)
    res = LICP(data, alpha=0.1)

    if len(res) == 0:
        skipped += 1
        continue

    if np.sum(res) == 0:
        all_false += 1

    is_subset = np.all(~res | ground_truth)

    # print(f"{i}: {res} - is subset: {is_subset}")    

    if not is_subset:
        bad += 1
    

print(f"False Positive: {bad / (total - skipped)}")
print(f"False Negative: {all_false / (total - skipped)}")

False Positive: 0.2336448598130841
False Negative: 0.5233644859813084
